In [22]:
from dotenv import load_dotenv
import os

load_dotenv()
apikey = os.getenv("OPENAI_API_KEY")
default_model = os.getenv("OPENAI_DEFAULT_MODEL")
g_apikey = os.getenv("GEMINI_API_KEY")
g_default_model = os.getenv("GEMINI_DEFAULT_MODEL")

if not apikey:
    raise RuntimeError("OPENAI_API_KEY is not set. Add it to the .env file.")
if not default_model:
    raise RuntimeError("OPENAI_DEFAULT_MODEL is not set. Add it to the .env file.")

default_model = default_model.strip().lower()
print(f"Using model: {default_model}")
print(f"Gemini API key configured: {bool(g_apikey)}")
print(f"Gemini model: {g_default_model}")

Using model: gpt-6-luna
Gemini API key configured: True
Gemini model: gemini-3.5-flash-lite


In [23]:
from openai import OpenAI

question = '오늘서울날씨 어때?'

client = OpenAI(api_key=apikey)

response = client.responses.create(
    model=default_model,
    tools=[{'type':'web_search'}],
    input = question
)

print(response.output_text)

오늘(10월 13일) 서울은 현재 비가 오고, 기온은 약 **16°C**예요. 오전엔 비가 오다가 점차 개겠고, 낮 최고 **22°C**, 최저 **9°C**로 예보돼요. 


In [24]:
# 1. Vector Store 생성
vector_store = client.vector_stores.create(
    name="PDF 저장소"
)
# 2. a.pdf 업로드 + Vector Store에 등록
file_batch = client.vector_stores.file_batches.upload_and_poll(
    vector_store_id=vector_store.id,
    files=[open("./Hotel Service Manual.pdf", "rb")]
)
print("업로드 상태:", file_batch.status)
print("파일 개수:", file_batch.file_counts)
# 3. File Search로 PDF 검색/요약
response = client.responses.create(
    model=default_model,
    tools=[{
            "type": "file_search",
            "vector_store_ids": [vector_store.id]
        }],
    input="업로드한 PDF를 5줄로 요약해줘."
)
print(response.output_text)

업로드 상태: completed
파일 개수: FileCounts(cancelled=0, completed=1, failed=0, in_progress=0, total=1)
- 고객을 친절하고 개인화된 인사로 맞이하고, 체크인·체크아웃은 각각 15시·11시 기준으로 안내합니다. 
- 객실 무료 Wi‑Fi와 180대 무료 주차 정보를 정확히 제공하고, 장애인 편의시설 이용 고객에게 필요한 도움을 줍니다. 
- 반려동물은 동반할 수 없음을 정중히 알리고, 필요하면 인근 반려동물 동반 가능 숙소를 안내합니다. 
- 룸서비스는 23시까지 운영하며, 객실은 금연이고 흡연실은 1층에 있습니다. 
- 취소 수수료는 전날 30%, 당일 50%, 노쇼 100%이며, 다양한 결제 수단을 안내하고 모든 고객을 존중합니다. 


In [25]:
# 1. Vector Store 생성
vector_store = client.vector_stores.create(
    name="PDF 저장소"
)
# 2. a.pdf 업로드 + Vector Store에 등록
file_batch = client.vector_stores.file_batches.upload_and_poll(
    vector_store_id=vector_store.id,
    files=[open("./투운사 정리본.docx", "rb")]
)
print("업로드 상태:", file_batch.status)
print("파일 개수:", file_batch.file_counts)
# 3. File Search로 PDF 검색/요약
response = client.responses.create(
    model=default_model,
    tools=[{
            "type": "file_search",
            "vector_store_ids": [vector_store.id]
        }],
    input="업로드한 docx를 5줄로 요약해줘."
)
print(response.output_text)

업로드 상태: completed
파일 개수: FileCounts(cancelled=0, completed=1, failed=0, in_progress=0, total=1)
- 문서는 투자운용사 시험 대비 정리본으로, 세금·절세 전략과 금융상품의 주요 개념을 다룹니다. 
- 포트폴리오 성과 측정 지표(샤프·트레이너·젠센·정보비율)와 금액가중·시간가중 수익률의 차이를 정리합니다. 
- 자산배분과 패시브 운용, 벤치마크 및 국제주가지수 등 투자전략의 기초를 설명합니다. 
- 기술적·기업가치 분석, 위험 측정(VaR·부도위험), 헤지펀드 전략과 부동산 금융상품도 요약합니다. fileciteturn0file2turn0file12turn0file15
- 집합투자기구의 구조·운용 제한·환매 절차와 금융 관련 규정 등 시험에 필요한 내용을 핵심 위주로 모았습니다. fileciteturn0file11turn0file18


In [17]:
#실제 코딩된 mock data만 반환하기

def get_weather(location):    
    """지정한 도시의 날씨 정보를 반환합니다.""" # DocString 설정
    weather_data = {
        "서울": "맑고 기온은 24도입니다.",
        "뉴욕": "비가 오고 있으며 기온은 15도입니다.",
        "도쿄": "흐림, 기온은 20도입니다."
    }    
    return weather_data.get(location, f'{location}의 날씨 정보가 없습니다')

In [29]:
weather_tools = [{
        "type": "function",
        "name": "get_weather",
        "description": "지정한 도시의 현재 날씨를 조회한다.",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "서울"
                }
            },
            "required": ["location"],
            "additionalProperties": False,
        },
        "strict": True,
}]

In [30]:
# help(get_weather)
help(str.upper)

Help on method_descriptor:

upper(self, /) unbound builtins.str method
    Return a copy of the string converted to uppercase.



In [37]:
tool_map = {"get_weather": get_weather}

In [32]:
# 1차 LLM 호출

city = "서울"
input_messages = f"{city}의 날씨는 어때?"
#input_messages = "바나나는 무슨 색깔이니?"

# 1) 1차 요청 : 모델에게 먼저 물어봄 (모델이 tool call을 할 수도, 바로 답할 수도)
resp1 = client.responses.create(
    model=default_model,
    input=input_messages,
    tools=weather_tools,
    tool_choice="auto"
)

In [34]:
for item in response.output:
    print('type', item.type)
    if item.type == 'function_call':
        print(item.name)
        print(item.arguments)
        print(item.call_id)

type file_search_call
type message


In [35]:
print(resp1.output)
print(resp1.output[0].call_id)
print(resp1.output[0].name)
print(resp1.output[0].type)
print(resp1.output[0].arguments)

[ResponseFunctionToolCall(arguments='{"location":"서울"}', call_id='call_wUrRll5HD4C7JXqbW9YrtmiA', name='get_weather', type='function_call', id='fc_01ffae8f19dbf42f006acd8488aeec87d08aebe35c920cef8b', async_=None, caller=None, namespace=None, status='completed')]
call_wUrRll5HD4C7JXqbW9YrtmiA
get_weather
function_call
{"location":"서울"}


In [42]:
import json

# 2) 모델이 요청한 함수 실행 결과를 만든다.
tool_outputs = []
for item in resp1.output:
    if item.type != "function_call":
        continue

    arguments = json.loads(item.arguments)
    function = tool_map[item.name]
    # print(arguments)
    # print(function)
    
    result = function(**arguments)

    tool_outputs.append(
        {
            "type": "function_call_output",
            "call_id": item.call_id,
            "output": result,
        }
    )
    print(tool_outputs)

    print(result)

[{'type': 'function_call_output', 'call_id': 'call_wUrRll5HD4C7JXqbW9YrtmiA', 'output': '맑고 기온은 24도입니다.'}]
맑고 기온은 24도입니다.


In [45]:
if not tool_outputs:
    print(response.output_text)
else :
    response2 = client.responses.create(
        model=default_model,
        input=tool_outputs,
        tools=weather_tools,
        previous_response_id = resp1.id
    )
    print(response2.output_text)


서울은 현재 맑고, 기온은 24도예요.


In [56]:
# 입력 -> 처리 -> 출력
# LangChain의 기본 : 프롬프트 -> LLM -> OutputParser
# from langchain_openai import ChatOpenAI
from langchain.chat_models import init_chat_model
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. 프롬프트 설정
prompt = ChatPromptTemplate.from_template('{location} 맛집 {count}곳만 알려줘') #서울 맛집 3곳, 부산 맛집 2곳

# 2. LLM (Large Language Model) 설정 
o_llm =init_chat_model('openai:gpt-6-luna')
g_llm = init_chat_model('google_genai:gemini-3.5-flash-lite')

# 3. 결과 생성
parser = StrOutputParser()

# 러너블 인터페이스
chain = prompt | g_llm | parser
chain2 = prompt | o_llm | parser
result = chain.invoke({'location': '서울', 'count': 3})
print(response)
print('---------------------------')
result = chain2.invoke({'location': '부산', 'count': 2})
print(response)

Response(id='resp_08136a0a1d0fd6d7006acd83e2f35087d0a4541831763b8bf7', access_programs=None, created_at=1791853539.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-6-luna', object='response', output=[ResponseFileSearchToolCall(id='fs_08136a0a1d0fd6d7006acd83e39cec87d0be528b22668ce006', queries=['업로드한 docx 문서의 핵심 내용과 주요 결론을 5줄로 요약해줘.'], status='completed', type='file_search_call', results=None), ResponseOutputMessage(id='msg_08136a0a1d0fd6d7006acd83e5819c87d0a98cae6afde0924f', content=[ResponseOutputText(annotations=[AnnotationFileCitation(file_id='file-2WrX89wBJqpfxCci45hJMS', filename='투운사 정리본.docx', index=54, type='file_citation'), AnnotationFileCitation(file_id='file-2WrX89wBJqpfxCci45hJMS', filename='투운사 정리본.docx', index=116, type='file_citation'), AnnotationFileCitation(file_id='file-2WrX89wBJqpfxCci45hJMS', filename='투운사 정리본.docx', index=166, type='file_citation')], text='- 문서는 투자운용사 시험 대비 정리본으로, 세금·절세 전략과 금융상품의 주요 개념을 다룹니다. \n- 포트폴리오 성과 측정 지표(샤프

In [62]:

# 입력 -> 처리 -> 출력
# LangChain의 기본: 프롬프트 -> LLM -> OutputParser

from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. 프롬프트 설정
prompt = ChatPromptTemplate.from_template(
    '{location} 맛집 {count}곳만 알려줘'
)

# 2. LLM 설정
o_llm = init_chat_model('openai:gpt-6-luna')
g_llm = init_chat_model('google_genai:gemini-3.5-flash-lite')

l_llm = init_chat_model(
    'openai:gemma-3-1b-it',
    base_url='http://localhost:1235/v1',
    api_key='not-needed'
)

# LM Studio 로컬 서버 설정
# Base URL: http://localhost:1235/v1
# Model: gemma-3-1b-it

# 3. 출력 파서 설정
parser = StrOutputParser()

# 4. 체인 연결: 프롬프트 -> LLM -> 파서
chain = prompt | g_llm | parser
chain2 = prompt | o_llm | parser
chain3 = prompt | l_llm | parser

# 5. 실행 및 출력
result = chain.invoke({'location': '서울', 'count': 3})
print(result)

print('---------------------------')

result = chain2.invoke({'location': '부산', 'count': 2})
print(result)

print('---------------------------')

result = chain3.invoke({'location': '경주', 'count': 2})
print(result)


서울에서 워낙 유명하고 매력적인 맛집이 많아 고르기 어렵지만, **서로 다른 매력(한식, 고기, 양식)**을 가진 대표 맛집 3곳을 추천해 드립니다.

---

### 1. 몽탄 (삼각지) - 우대갈비의 성지
* **특징:**짚불에 향을 입힌 두툼한 '우대갈비'로 유명한 곳입니다. 은은한 훈연 향과 부드러운 육질이 일품이며, 고기를 다 먹은 후 불판에 볶아주는 '양파볶음밥'과 '된장찌개'도 극찬을 받는 메뉴입니다.
* **대표 메뉴:** 우대갈비, 짚불삼겹살, 양파볶음밥
* **팁:** 웨이팅이 매우 극심하므로 오픈 전 현장 웨이팅 등록이나 캐치테이블 앱을 통한 예약 전략이 필요합니다.

### 2. 남영돈 (남영역/숙대입구) - 대한민국 최고 수준의 목살
* **특징:** 미식가들 사이에서 "목살은 여기가 최고"라고 입을 모아 말하는 돼지고기 전문점입니다. 참숯 화로에 구워 육즙이 가득 갇힌 가브리살과 항정살, 목살을 맛볼 수 있습니다. 
* **대표 메뉴:** 항정살, 가브리살, 목살, 비빔냉면
* **팁:** 고기를 처음부터 끝까지 직원분이 가장 맛있게 구워주어 편하게 즐길 수 있습니다. 이곳 역시 웨이팅이 긴 편입니다.

### 3. 유림면 (시청/덕수궁) - 60년 전통의 메밀국수
* **특징:** 미쉐린 가이드 서울에 꾸준히 이름을 올리고 있는 노포입니다. 깔끔하고 깊은 맛의 육수에 말아 나오는 메밀국수와 바삭한 비빔국수, 그리고 겨울철에 특히 좋은 뜨끈한 국물의 냄비국수가 유명합니다.
* **대표 메뉴:** 메밀국수, 비빔메밀, 냄비국수
* **팁:** 회전율이 비교적 빠른 편이라 웨이팅이 있어도 줄이 금방 줄어드는 편입니다. 덕수궁 산책 코스와 함께 묶기 좋습니다.

---

*※ 세 곳 모두 워낙 인기 있는 곳이라 방문 전 영업시간이나 브레이크 타임을 꼭 확인하고 가시는 것을 추천합니다!*
---------------------------
1. **해운대암소갈비집** — 해운대의 대표 갈비 맛집. 양념갈비가 인기예요.  
2. **쌍둥이돼지국밥

In [68]:
# invoke(1:1)
# 1. 프롬프트 설정

prompt = ChatPromptTemplate.from_template(
    '{location} 맛집 {count}곳만 알려줘'
)

# 2. LLM 설정
llm = init_chat_model('openai:gpt-6-luna')

# 3. 출력 파서 설정
parser = StrOutputParser()

# 4. 체인 연결: 프롬프트 -> LLM -> 파서
chain = prompt | llm | parser

# 5. 실행 및 출력
result = chain.invoke({'location': '서울', 'count':1})

In [63]:
result = await chain.ainvoke({'location': '서울', 'count':1})
print(result)

서울 중구(을지로)에 위치한 **'우래옥'**을 추천합니다.

*   **메뉴:** 평양냉면, 불고기
*   **특징:** 1946년부터 이어져 온 전통 있는 평양냉면 전문점으로, 진하고 묵직한 육향이 일품인 육수와 메밀향 가득한 면의 조화가 환상적입니다. 냉면 외에도 달콤 짭조름한 불고기를 곁들여 먹기 좋습니다.
*   **한 줄 평:** 서울에서 제대로 된 평양냉면과 노포의 깊은 세월을 느끼고 싶다면 반드시 가봐야 할 곳입니다.


In [65]:
response = chain.astream({'location': '서울', 'count':2})

async for chunk in response:
    print(chunk, end="", flush=True)

서울에서 정말로 실패 없는 맛집 2곳을 추천해 드립니다. 각각 다른 매력의 장소입니다.

**1. 남영돈 (서울 용산구 - 삼겹살/목살)**
*   **특징:** 대한민국에서 가장 줄 서서 먹는 돼지고기 집 중 하나입니다. 참숯 화로에 숯 향을 듬뿍 입혀 구워주는 목살과 삼겹살이 예술입니다.
*   **추천 메뉴:** 항정살, 목살 (육즙이 가득하고 엄청 부드럽습니다.)
*   **팁:** 웨이팅이 매우 극심하므로 오픈 전 미리 가서 예약하거나 테이블링 앱을 확인해야 합니다.

**2. 몽탄 (서울 용산구 - 우대갈비)**
*   **특징:** 웨이팅 전쟁으로 유명한 짚불 구이 전문점입니다. 짚불에 초벌한 우대갈비의 풍미가 독보적이고, 화려한 비주얼만큼 맛도 훌륭합니다.
*   **추천 메뉴:** 우대갈비, 볶음밥(양파볶음밥)
*   **팁:** 예약 시스템이나 현장 대기 등록 방식을 미리 확인하고 방문하시는 것을 추천합니다.

두 곳 모두 용산(삼각지·남영) 근처라 일정 잡기에도 좋습니다!

In [69]:
batch_messages = [
    [("system", "당신은 서울의 음식 전문가입니다."), ("user", "광장시장에서 먹을 만한 길거리 음식을 소개해 주세요.")],
    [("system", "당신은 서울의 문화 전문가입니다."), ("user", "서울에서 데이트할 때 가 볼 만한 곳을 알려 주세요.")],
]

responses = llm.batch(batch_messages)

for item in responses:
    print(item.content)
    print('-'*30)

광장시장에 처음 가신다면 아래 메뉴를 추천합니다.

- **빈대떡**: 녹두를 갈아 부쳐 겉은 바삭하고 속은 고소합니다. 김치나 양파절임을 곁들이면 느끼함이 덜해요.
- **마약김밥**: 단무지와 당근 등을 넣은 작고 담백한 김밥으로, 겨자 간장에 찍어 먹는 게 별미입니다.
- **떡볶이·순대·어묵**: 시장 분식의 기본 조합이에요. 매콤한 떡볶이 국물에 순대나 어묵을 곁들이면 든든합니다.
- **육회 또는 육회비빔밥**: 길거리 간식이라기보다는 시장의 유명한 한 끼 메뉴입니다. 생고기를 먹을 수 있을 때 골라 보세요.
- **칼국수**: 따뜻하고 푸짐한 음식이 당길 때 좋습니다. 붐비는 시간에는 자리가 꽤 복잡할 수 있어요.

**처음이라면 빈대떡과 마약김밥을 나눠 먹고, 배가 더 고프면 떡볶이나 칼국수를 추가하는 조합**이 무난합니다. 시장 안은 붐비니 주문 전에 양과 가격을 확인하고, 여러 가게에서 조금씩 맛보면 더 다양하게 즐길 수 있어요.
------------------------------
서울은 **산책·전시·맛집을 한 동네에서 함께 즐기기 좋아** 데이트 코스를 짜기 편합니다. 분위기별로 추천해 드릴게요.

### 걷고 구경하기 좋은 곳
- **서촌·경복궁**: 경복궁이나 국립현대미술관 서울관을 둘러본 뒤, 서촌 골목의 작은 카페와 식당에 들러 보세요. 조용하고 차분한 데이트에 잘 어울립니다.
- **익선동·종로**: 한옥 골목을 구경하고 식사나 디저트를 즐기기 좋습니다. 사람이 붐비는 편이라 주말보다는 평일 저녁이 한결 여유로워요.
- **서울숲·성수**: 서울숲 산책 후 성수의 전시 공간이나 카페를 방문하는 코스입니다. 구경할 곳이 많아 취향에 따라 일정을 유연하게 바꾸기 좋습니다.

### 야경이나 분위기를 즐기고 싶다면
- **남산서울타워**: 남산 산책과 전망대 야경을 함께 즐길 수 있습니다. 날씨가 맑은 날에 특히 좋고, 해 질 무렵에 올라가면 낮과 밤의 풍경을 모두 볼 수 있어요.
- **한강공원**: 반포·여의도·뚝섬 중 동선에 

In [70]:
import asyncio
import time
question1 = [
    ("system", "당신은 서울의 음식 전문가입니다."),
    ("human", "광장시장 대표 음식 한 가지를 한 문장으로 알려 주세요."),
]
question2 = [
    ("system", "당신은 서울의 음식 전문가입니다."),
    ("human", "남대문 시장 대표 음식 한 가지를 한 문장으로 알려 주세요."),
]
question3 = [
    ("system", "당신은 서울의 음식 전문가입니다."),
    ("human", "동대문 시장 대표 음식 한 가지를 한 문장으로 알려 주세요."),
]


In [71]:
%%time
result1 = llm.invoke(question1)
result2 = llm.invoke(question2)
result3 = llm.invoke(question3)

CPU times: total: 31.2 ms
Wall time: 6.14 s


In [80]:
r1, r2, r3 = await asyncio.gather(
    llm.ainvoke(question1),
    llm.ainvoke(question2),
    llm.ainvoke(question3)
)

In [ ]:
[('system', 'content'),('user','contetn1'),('assistant','content2')]

[('system', '너는 여행 전문가야'),
 ('user', '서울에서 가볼 만한 여행지를 추천해 주세요.'),
 
 ('assistant', '서울에서 가볼 만한 여행지로는 경복궁'),
 
 ('user', '서울역에서 그 곳까지 가는 대중교통수단 알려줘'),
 
 ('assistant', '서울역에서 경복궁까지 가는 대중교통수단은 지하철 1호선을 이용하는 것이 가장 편리합니다. 서울역에서 1호선을 타고 종로3가역에서 하차한 후, 3번 출구로 나와 도보로 약 10분 정도 걸어가면 경복궁에 도착할 수 있습니다.')]

[('system', '너는 여행 전문가야'),
 ('user', '서울에서 가볼 만한 여행지를 추천해 주세요.'),
 ('assistant', '서울에서 가볼 만한 여행지로는 경복궁'),
 ('user', '서울역에서 그 곳까지 가는 대중교통수단 알려줘')]

In [ ]:
messages = [('system' , '당신은 여행 전문가로 고객의 여행 일정에 도움을 줍니다'),
            ('user', '부산 여행에서 딱 한 곳만 가봐야 한다면 어떤 곳인지 알려주세요')]
ret = llm.invoke(messages)

messages.append(('assistant', ret.content))

print(messages)
messages.append(('user', '부산역에서 그곳에 가는 교통편을 알려주세요'))

print(messages)
messages.append(('user', '부산역에서 그곳에 가는 교통편을 알려주세요'))

딱 한 곳만 고른다면 **광안리해변**을 추천해요. 바다 너머로 광안대교가 펼쳐져 부산다운 풍경을 한눈에 볼 수 있고, 해 질 무렵부터 밤까지 분위기가 특히 좋습니다. 해변을 걷고 주변 카페나 식당에 들르기도 편해요.

가능하면 **해질 무렵에 도착해 야경까지** 즐겨보세요.


In [ ]:
from langchain_core.prompts import ChatPromptTemplate

ChatPromptTemplate.from_template('카레의 기본 레시피를 재료와 함께 알려주세요')    
# from_template : 단순 문자열
# from_messages : 역할 지정 : system, user, assistant

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_template('{food}의 기본 레시피를 재료와 함께 알려주세요')    

chain = prompt | llm | StrOutputParser()

ret = chain.invoke({'food': '카레'})
print(ret)


## 기본 카레 레시피 (3~4인분)

### 재료
- 밥 3~4공기
- 카레용 고형 카레 4~5조각 *(제품에 따라 조절)*
- 돼지고기나 소고기 200g
- 감자 1개
- 양파 1개
- 당근 ½개
- 물 600~700ml
- 식용유 약간
- 소금·후추 약간

### 만드는 법
1. 고기와 감자, 양파, 당근을 한입 크기로 썹니다.
2. 냄비에 식용유를 두르고 양파를 볶다가 고기를 넣어 겉면이 익을 때까지 볶습니다.
3. 감자와 당근을 넣고 2~3분 더 볶은 뒤 물을 붓습니다.
4. 끓어오르면 약한 불로 줄이고, 감자와 당근이 익을 때까지 15~20분 끓입니다.
5. 불을 끄거나 아주 약하게 줄인 뒤 고형 카레를 넣고 잘 풀어줍니다.
6. 다시 약불에서 5분 정도 저어가며 끓입니다. 밥 위에 카레를 얹어 냅니다.

카레가 너무 되직하면 물을 조금 더 넣고, 묽으면 약불에서 더 끓여 농도를 맞추세요.


In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_template('{food}의 기본 레시피를 재료와 함께 알려주세요')    

chain = prompt | llm | StrOutputParser()

ret = chain.invoke({'food': '카레'})
print(ret)


In [88]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_template('{baseball}의 유명 팀의 이름 3개를 알려주세요')    

chain = prompt | llm | StrOutputParser()

ret = chain.invoke({'baseball': '야구'})
print(ret)




유명한 야구팀 세 곳은 다음과 같습니다.

- 뉴욕 양키스
- 로스앤젤레스 다저스
- 보스턴 레드삭스


In [91]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_template('{age}을(를) 대상으로 하는 {food}의 홍보 슬로건 3가지 작성, 톤앤매너는 {tone}으로')    

chain = prompt | llm | StrOutputParser()

ret = chain.invoke({'age': '2030 직장인', 'food': '무설탕 디저트', 'tone': '위트있고 감각적인'})
print(ret)

1. **당 충전은 빼고, 오후의 행복은 더하고.**
2. **회의는 달콤하지 않아도, 디저트는 가볍게 달콤하게.**
3. **퇴근 전 한 입, 죄책감은 로그아웃.**


In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 주석을 추가할 원본 파이썬 코드입니다.
code = """numbers = [1, 2, 3, 4, 5]
squared = [number ** 2 for number in numbers]
print(squared)"""

# 코드의 동작과 출력은 유지하면서, 초보자에게 친절한 한국어 주석만 추가하도록 요청합니다.
prompt = ChatPromptTemplate.from_template(
    """다음 파이썬 코드에 {python_dev}이 이해하기 쉽도록 {tone} 말투의 한국어 주석을 추가해 주세요.
코드의 실행 결과와 동작은 바꾸지 말고, 주석이 추가된 전체 코드만 반환해 주세요.

{code}"""
)

# 프롬프트 -> LLM -> 문자열 출력 파서 순서로 체인을 구성합니다.
chain = prompt | llm | StrOutputParser()

# 원본 코드, 대상 독자, 주석 말투를 전달해 주석이 추가된 코드를 생성합니다.
ret = chain.invoke({
    'python_dev': '파이썬 초보 개발자',
    'tone': '친절하고 이해하기 쉬운',
    'code': code,
})
print(ret)

In [93]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 서울의 음식과 문화 전문가입니다."),
    MessagesPlaceholder("history"),   # 이전 대화가 들어갈 자리
    ("human", "{question}"),
])

chain = prompt | llm | StrOutputParser()

history = []

# 1턴
q1 = "서울을 대표하는 음식 하나만 알려줘."
a1 = chain.invoke({"history": history, "question": q1})
history += [('user', q1), ('assistant', a1)]

# 2턴: 앞의 대화를 기억하고 답변
q2 = "그 음식을 먹을 수 있는 식당도 추천해줘."
a2 = chain.invoke({"history": history, "question": q2})
history += [('user', q2), ('assistant', a2)]
print(a2)

**이문설농탕**을 추천해요. 종로에 있는 오래된 설렁탕집으로, 진한 사골 국물과 담백한 맛이 특징입니다. 종로 일대를 둘러본 뒤 들르기 좋아요. 영업시간이나 위치가 바뀔 수 있으니 방문 전에 확인해 보세요.


In [ ]:
# prompt | llm | parser
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import ChatPromptTemplate 
from pydantic import BaseModel

class PersonInfo(BaseModel):
    name: str
    age: int 
    
parser = PydanticOutputParser(pydantic_object=PersonInfo)

query = "홍길동은 올해 20살 대학생입니다."
query1 = "김길동은 올해 30살 대학생입니다"

prompt = ChatPromptTemplate.from_template(
    """
    다음 문장을 분석하여, 이름, 나이를 추출하세요
    문장:{text}
    {format_instructions} 
    """
).partial(format_instructions = parser.get_format_instructions())

chain = prompt | llm | parser

result = chain.invoke({"text": query,})
# result = chain.invoke({"text": query1, 'format_instructions': parser.get_format_instructions()})

In [96]:
print(result)

name='홍길동' age=20


In [97]:
# prompt | llm | parser
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import ChatPromptTemplate 
from pydantic import BaseModel

class PersonInfo(BaseModel):
    name: str
    age: int 
    
parser = PydanticOutputParser(pydantic_object=PersonInfo)

query = "홍길동은 올해 20살 대학생입니다."
query1 = "김길동은 올해 30살 대학생입니다"

prompt = ChatPromptTemplate.from_template(
    """
    다음 문장을 분석하여, 이름, 나이를 추출하세요
    문장:{text}
    
    """
)

s_llm = llm.with_structured_output(PersonInfo)
chain = prompt | s_llm 
result = chain.invoke({"text": query1})
print(result)
# result = chain.invoke({"text": query1, 'format_instructions': parser.get_format_instructions()})

name='김길동' age=30
